[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Leonardo-Baptista/aula_SAM_propemm/blob/main/notebooks/02_treinar_no_colab.ipynb)


# Treinar no Colab com a exportação do sam-anota

1. No sam-anota, clique em **Exportar...** e marque os formatos. A saída fica em `saidas/<pasta>/treino_<data e hora>/`.
2. Compacte essa pasta num `.zip` (Windows: botão direito, Enviar para, Pasta compactada).
3. Aqui no Colab: Ambiente de execução, Alterar o tipo de ambiente de execução, **GPU T4**.
4. Rode a célula de envio e depois **uma** das opções A, B ou C.

Cada pasta da exportação tem um `README_TREINO.md` com os mesmos comandos.

In [ ]:
# Envio do zip pelo navegador (ou monte o Drive e aponte RAIZ para a pasta)
import pathlib, zipfile
from google.colab import files

enviado = files.upload()  # escolha treino_AAAAMMDD_HHMMSS.zip
nome = next(iter(enviado))
zipfile.ZipFile(nome).extractall("dados_treino")
RAIZ = next(p.parent for p in pathlib.Path("dados_treino").rglob("divisao.json")).resolve()
print(RAIZ)
print(sorted(p.name for p in RAIZ.iterdir()))

## A) YOLO-seg (Ultralytics YOLO11 ou YOLO26)

A Ultralytics é AGPL 3.0; por isso fica fora do sam-anota e é instalada só aqui. O treino roda de dentro da pasta `yolo/` (o `data.yaml` usa `path: .`).

In [ ]:
!pip -q install ultralytics
%cd {RAIZ / "yolo"}
!yolo segment train data=data.yaml model=yolo11n-seg.pt epochs=50 imgsz=640
# YOLO26: troque para model=yolo26n-seg.pt
%cd /content

## B) Mask R-CNN do torchvision (formato COCO)

O torch com CUDA já vem no Colab. O script baixa os pesos COCO do Mask R-CNN v2 (cerca de 180 MB).

In [ ]:
!pip -q install pycocotools
%cd {RAIZ / "coco"}
!python treinar_torchvision.py --iteracoes 300
%cd /content

## C) U-Net (máscaras semânticas)

Usa `segmentation_models_pytorch` com encoder resnet34 pré-treinado no ImageNet.

In [ ]:
!pip -q install segmentation-models-pytorch
%cd {RAIZ / "semantica"}
!python treinar_unet.py --iteracoes 500 --tamanho 512
%cd /content

## Detectron2

O Detectron2 não tem pacote oficial no pip; no Colab ele precisa ser compilado (alguns minutos). O trecho de treino com `register_coco_instances` e `mask_rcnn_R_50_FPN_3x` está em `coco/README_TREINO.md`. Use `cfg.INPUT.MASK_FORMAT = "bitmask"`, porque objetos com furo ou finos (trincas) vêm em RLE.